# 68. The AS/RS Task Interleaving Problem
## Tier 9: The Quantum Leap (Quantum Annealing / QUBO)

### Key Assumptions
- Quadratic Unconstrained Binary Optimization (QUBO) formulation for quantum annealing
- Binary variables represent task sequence decisions: $x_{i,j} = 1$ if task $i$ precedes task $j$
- QUBO objective: $\min \sum_{i,j} w_{i,j} x_{i,j} + \lambda \sum_{i,j,k} (x_{i,j} + x_{j,k} - x_{i,k})^2$
- Quantum annealing hardware simulation with D-Wave Ocean SDK
- Penalty constraints for sequence validity and task ordering
- Hybrid quantum-classical optimization approach

### Approach (Step-by-Step)
1. **QUBO Formulation**: Convert task sequencing problem to binary optimization
2. **Distance Matrix**: Calculate Manhattan distances between all task locations
3. **Constraint Encoding**: Add penalty terms for sequence validity
4. **Quantum Annealing**: Solve QUBO using simulated quantum annealing
5. **Solution Decoding**: Convert binary solution back to task sequence

### What to Look for in the Results
- QUBO formulation correctness and constraint satisfaction
- Quantum annealing convergence and solution quality
- Comparison with classical optimization methods
- Performance scaling with problem size
- Quantum advantage demonstration (if applicable)
- Solution feasibility and optimality verification

### Concrete Example (from the source)
Quantum annealing for 6-task AS/RS problem:

**QUBO Configuration**:
- Binary variables: 36 (6×6 sequence matrix)
- Penalty weight λ: 1000 for constraint enforcement
- Annealing schedule: 1000 sweeps with exponential cooling
- Chain strength: 2.0 for variable coupling

**Expected Results**:
- Optimal sequence: ['S2', 'R2', 'S1', 'R1', 'S3', 'R3']
- Total travel time: 39.8 seconds
- QUBO objective value: -39.80
- Constraint violations: 0 (feasible solution)
- Quantum speedup: 15% faster than classical methods
- Solution quality: 98% of optimal benchmark

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries for Quantum Annealing
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import time
from collections import defaultdict
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Set up professional visualization style
plt.style.use('default')
sns.set_palette("husl")

# Note: D-Wave Ocean SDK would be used for real quantum annealing
# Here we implement a simulated quantum annealing approach
print("Note: This implementation uses simulated quantum annealing.")
print("For real quantum hardware, install: pip install dwave-ocean-sdk")

Note: This implementation uses simulated quantum annealing.
For real quantum hardware, install: pip install dwave-ocean-sdk


In [2]:
class QuantumASRS:
    """
    Quantum Annealing solver for AS/RS Task Interleaving.
    """
    
    def __init__(self, tasks, depot=(1,1)):
        """
        Initialize quantum AS/RS optimizer.
        
        Args:
            tasks: List of (id, x, y, priority) tuples
            depot: Starting position (x, y)
        """
        self.tasks = tasks
        self.depot = depot
        self.num_tasks = len(tasks)
        
        # Calculate distance matrix
        self.distance_matrix = self._calculate_distance_matrix()
        
        # QUBO parameters
        self.penalty_weight = 1000.0  # λ for constraint enforcement
        self.annealing_schedule = {
            'initial_temp': 10.0,
            'final_temp': 0.01,
            'num_sweeps': 1000,
            'beta_schedule': 'exponential'
        }
        
        # Solution tracking
        best_solution = None
        best_energy = float('inf')
        energy_history = []
    
    def _calculate_distance_matrix(self):
        """
        Calculate Manhattan distance matrix including depot.
        
        Returns:
            distance_matrix: Matrix of distances between all locations
        """
        # Include depot as first location
        locations = [self.depot] + [(task[1], task[2]) for task in self.tasks]
        n = len(locations)
        
        distance_matrix = np.zeros((n, n))
        
        for i in range(n):
            for j in range(n):
                if i != j:
                    dist = max(abs(locations[i][0] - locations[j][0]), 
                             abs(locations[i][1] - locations[j][1]))
                    distance_matrix[i][j] = dist
        
        return distance_matrix
    
    def create_qubo(self):
        """
        Create QUBO formulation for task sequencing.
        
        Returns:
            Q: QUBO matrix (quadratic terms)
            linear: Linear terms
        """
        n = self.num_tasks
        total_vars = n * n  # x[i,j] variables
        
        # Initialize QUBO matrices
        Q = np.zeros((total_vars, total_vars))
        linear = np.zeros(total_vars)
        
        # Helper function to get variable index
        def var_index(i, j):
            return i * n + j
        
        # Objective: Minimize total travel distance
        for i in range(n):
            for j in range(n):
                if i != j:
                    # Distance from depot to first task
                    if i == 0:
                        idx = var_index(i, j)
                        linear[idx] += self.distance_matrix[0, j+1] * 0.5
                    # Distance between tasks
                    else:
                        idx = var_index(i, j)
                        for k in range(n):
                            if k != i and k != j:
                                idx2 = var_index(k, i)
                                Q[idx, idx2] += self.distance_matrix[j+1, k+1] * 0.25
                    # Distance from last task back to depot
                    if j == n-1:
                        idx = var_index(i, j)
                        linear[idx] += self.distance_matrix[j+1, 0] * 0.5
        
        # Constraint 1: Each position has exactly one task
        for j in range(n):
            for i1 in range(n):
                for i2 in range(n):
                    if i1 != i2:
                        idx1 = var_index(i1, j)
                        idx2 = var_index(i2, j)
                        Q[idx1, idx2] += self.penalty_weight
            
            # Linear penalty for deviation from 1
            for i in range(n):
                idx = var_index(i, j)
                linear[idx] -= self.penalty_weight
            
            # Constant term
            linear[0] += self.penalty_weight  # Add to first variable (will be subtracted later)
        
        # Constraint 2: Each task appears exactly once
        for i in range(n):
            for j1 in range(n):
                for j2 in range(n):
                    if j1 != j2:
                        idx1 = var_index(i, j1)
                        idx2 = var_index(i, j2)
                        Q[idx1, idx2] += self.penalty_weight
            
            # Linear penalty for deviation from 1
            for j in range(n):
                idx = var_index(i, j)
                linear[idx] -= self.penalty_weight
        
        # Constraint 3: Maintain sequence consistency
        for i in range(n):
            for j in range(n):
                if i != j:
                    for k in range(n):
                        if k != i and k != j:
                            idx1 = var_index(i, j)
                            idx2 = var_index(j, k)
                            idx3 = var_index(i, k)
                            
                            # x[i,j] + x[j,k] - x[i,k] >= 0
                            Q[idx1, idx2] += self.penalty_weight * 0.25
                            Q[idx1, idx3] -= self.penalty_weight * 0.5
                            Q[idx2, idx3] -= self.penalty_weight * 0.5
                            linear[idx3] += self.penalty_weight * 0.25
        
        return Q, linear
    
    def calculate_energy(self, state, Q, linear):
        """
        Calculate energy of a given state.
        
        Args:
            state: Binary state vector
            Q: QUBO matrix
            linear: Linear terms
        
        Returns:
            energy: Total energy of the state
        """
        # Linear term
        energy = np.dot(linear, state)
        
        # Quadratic term
        energy += 0.5 * np.dot(state, np.dot(Q, state))
        
        return energy
    
    def metropolis_acceptance(self, current_energy, new_energy, temperature):
        """
        Metropolis acceptance criterion.
        
        Args:
            current_energy: Current state energy
            new_energy: Proposed state energy
            temperature: Current temperature
        
        Returns:
            accepted: Whether to accept the new state
        """
        if new_energy < current_energy:
            return True
        
        # Metropolis criterion
        delta = current_energy - new_energy
        probability = np.exp(delta / temperature) if temperature > 0 else 0
        
        return random.random() < probability
    
    def simulated_quantum_annealing(self, Q, linear):
        """
        Simulated quantum annealing algorithm.
        
        Args:
            Q: QUBO matrix
            linear: Linear terms
        
        Returns:
            best_state: Best binary state found
            best_energy: Energy of best state
            energy_history: History of energies
        """
        n_vars = len(linear)
        
        # Initialize random state
        current_state = np.random.randint(0, 2, n_vars)
        current_energy = self.calculate_energy(current_state, Q, linear)
        
        best_state = current_state.copy()
        best_energy = current_energy
        energy_history = [current_energy]
        
        # Annealing parameters
        initial_temp = self.annealing_schedule['initial_temp']
        final_temp = self.annealing_schedule['final_temp']
        num_sweeps = self.annealing_schedule['num_sweeps']
        
        # Temperature schedule
        if self.annealing_schedule['beta_schedule'] == 'exponential':
            temp_schedule = np.logspace(np.log10(initial_temp), np.log10(final_temp), num_sweeps)
        else:
            temp_schedule = np.linspace(initial_temp, final_temp, num_sweeps)
        
        # Main annealing loop
        for sweep in range(num_sweeps):
            temperature = temp_schedule[sweep]
            
            # Multiple proposals per sweep
            for _ in range(n_vars):
                # Propose new state by flipping random bits
                proposal_state = current_state.copy()
                num_flips = random.randint(1, min(3, n_vars // 2))
                
                for _ in range(num_flips):
                    flip_idx = random.randint(0, n_vars - 1)
                    proposal_state[flip_idx] = 1 - proposal_state[flip_idx]
                
                # Calculate energy of proposal
                proposal_energy = self.calculate_energy(proposal_state, Q, linear)
                
                # Metropolis acceptance
                if self.metropolis_acceptance(current_energy, proposal_energy, temperature):
                    current_state = proposal_state
                    current_energy = proposal_energy
                    
                    # Update best solution
                    if current_energy < best_energy:
                        best_state = current_state.copy()
                        best_energy = current_energy
            
            energy_history.append(current_energy)
            
            # Progress reporting
            if sweep % 100 == 0:
                print(f"Sweep {sweep}/{num_sweeps}: Energy = {current_energy:.2f}, Best = {best_energy:.2f}")
        
        return best_state, best_energy, energy_history
    
    def decode_solution(self, state):
        """
        Decode binary state to task sequence.
        
        Args:
            state: Binary state vector
        
        Returns:
            sequence: Task sequence
            valid: Whether solution is valid
        """
        n = self.num_tasks
        sequence = []
        valid = True
        
        # Extract sequence from state matrix
        for pos in range(n):
            task_found = False
            for task in range(n):
                idx = task * n + pos
                if state[idx] == 1:
                    if task_found:
                        valid = False  # Multiple tasks in same position
                    sequence.append(task)
                    task_found = True
            
            if not task_found:
                valid = False  # Empty position
        
        # Check for duplicate tasks
        if len(set(sequence)) != len(sequence):
            valid = False
        
        return sequence, valid
    
    def calculate_sequence_time(self, sequence):
        """
        Calculate total time for a task sequence.
        
        Args:
            sequence: Task sequence
        
        Returns:
            total_time: Total travel and operation time
        """
        total_time = 0
        current_pos = self.depot
        
        for task_idx in sequence:
            task = self.tasks[task_idx]
            task_pos = (task[1], task[2])
            
            # Travel time
            distance = max(abs(task_pos[0] - current_pos[0]), abs(task_pos[1] - current_pos[1]))
            total_time += distance * 0.5
            
            # Operation time
            total_time += 3 if task[0].startswith('S') else 2
            
            current_pos = task_pos
        
        # Return to depot
        distance = max(abs(self.depot[0] - current_pos[0]), abs(self.depot[1] - current_pos[1]))
        total_time += distance * 0.5
        
        return total_time
    
    def optimize(self):
        """
        Run quantum annealing optimization.
        
        Returns:
            best_sequence: Best task sequence found
            best_time: Total time for best sequence
            best_energy: QUBO energy of best solution
            energy_history: Optimization history
        """
        print("Creating QUBO formulation...")
        Q, linear = self.create_qubo()
        
        print(f"QUBO size: {len(linear)} variables")
        print(f"Non-zero QUBO elements: {np.count_nonzero(Q)}")
        print(f"Penalty weight: {self.penalty_weight}")
        
        print("\nStarting quantum annealing...")
        start_time = time.time()
        
        best_state, best_energy, energy_history = self.simulated_quantum_annealing(Q, linear)
        
        end_time = time.time()
        optimization_time = end_time - start_time
        
        print(f"\nQuantum annealing completed in {optimization_time:.2f} seconds")
        print(f"Best energy: {best_energy:.2f}")
        
        # Decode solution
        sequence, valid = self.decode_solution(best_state)
        
        if valid:
            total_time = self.calculate_sequence_time(sequence)
            print(f"Valid solution found!")
            print(f"Sequence: {[self.tasks[i][0] for i in sequence]}")
            print(f"Total time: {total_time:.2f} seconds")
        else:
            print("Warning: Invalid solution detected")
            # Try to repair solution
            sequence = self.repair_solution(best_state)
            total_time = self.calculate_sequence_time(sequence) if sequence else float('inf')
        
        return sequence, total_time, best_energy, energy_history
    
    def repair_solution(self, state):
        """
        Repair invalid solution.
        
        Args:
            state: Invalid binary state
        
        Returns:
            repaired_sequence: Repaired task sequence
        """
        n = self.num_tasks
        
        # Simple repair: use greedy approach
        remaining_tasks = list(range(n))
        sequence = []
        
        # Build sequence from state matrix
        for pos in range(n):
            best_task = None
            best_score = -1
            
            for task in remaining_tasks:
                idx = task * n + pos
                if state[idx] == 1:
                    score = 1.0
                    if best_score < score:
                        best_score = score
                        best_task = task
            
            if best_task is not None:
                sequence.append(best_task)
                remaining_tasks.remove(best_task)
            else:
                # Add remaining tasks in order
                if remaining_tasks:
                    sequence.append(remaining_tasks.pop(0))
        
        return sequence

In [ ]:
# Initialize the AS/RS problem for quantum optimization
storage_tasks = [
    ('S1', 2, 3, 5),  # (id, x, y, priority)
    ('S2', 6, 5, 3),
    ('S3', 8, 2, 4)
]

retrieval_tasks = [
    ('R1', 3, 4, 8),
    ('R2', 7, 3, 6),
    ('R3', 5, 7, 2)
]

all_tasks = storage_tasks + retrieval_tasks

# Create quantum optimizer instance
quantum_optimizer = QuantumASRS(all_tasks, depot=(1,1))

print("AS/RS Task Interleaving - Quantum Annealing Optimization")
print(f"Storage tasks: {len(storage_tasks)}")
print(f"Retrieval tasks: {len(retrieval_tasks)}")
print(f"Total tasks: {quantum_optimizer.num_tasks}")
print(f"Depot position: {quantum_optimizer.depot}")
print()

# Display task information
print("Task Details:")
for i, task in enumerate(all_tasks):
    task_type = "Storage" if task[0].startswith('S') else "Retrieval"
    print(f"  {i}: {task[0]}: {task_type} at ({task[1]},{task[2]}), priority={task[3]}")

# Display distance matrix
print("\nDistance Matrix:")
print("    Depot", end="")
for i, task in enumerate(all_tasks):
    print(f"  {task[0]:4s}", end="")
print()

locations = ['Depot'] + [task[0] for task in all_tasks]
for i, loc1 in enumerate(locations):
    print(f"{loc1:4s}", end="")
    for j, loc2 in enumerate(locations):
        if i == j:
            print(f"  {'-':4s}", end="")
        else:
            print(f"  {quantum_optimizer.distance_matrix[i][j]:4.1f}", end="")
    print()

AS/RS Task Interleaving - Quantum Annealing Optimization
Storage tasks: 3
Retrieval tasks: 3
Total tasks: 6
Depot position: (1, 1)

Task Details:
  0: S1: Storage at (2,3), priority=5
  1: S2: Storage at (6,5), priority=3
  2: S3: Storage at (8,2), priority=4
  3: R1: Retrieval at (3,4), priority=8
  4: R2: Retrieval at (7,3), priority=6
  5: R3: Retrieval at (5,7), priority=2

Distance Matrix:
    Depot  S1    S2    S3    R1    R2    R3  
Depot  -      2.0   5.0   7.0   3.0   6.0   6.0
S1     2.0  -      4.0   6.0   1.0   5.0   4.0
S2     5.0   4.0  -      3.0   3.0   2.0   2.0
S3     7.0   6.0   3.0  -      5.0   1.0   5.0
R1     3.0   1.0   3.0   5.0  -      4.0   3.0
R2     6.0   5.0   2.0   1.0   4.0  -      4.0
R3     6.0   4.0   2.0   5.0   3.0   4.0  -   


In [ ]:
try:
    # Run quantum annealing optimization
    print("\n" + "=" * 60)
    print("QUANTUM ANNEALING OPTIMIZATION")
    print("=" * 60)
    
    start_time = time.time()
    best_sequence, best_time, best_energy, energy_history = quantum_optimizer.optimize()
    end_time = time.time()
    
    total_optimization_time = end_time - start_time
    print(f"\nTotal optimization time: {total_optimization_time:.2f} seconds")
    print(f"Final energy: {best_energy:.2f}")
    print(f"Best sequence: {[all_tasks[i][0] for i in best_sequence]}")
    print(f"Total travel time: {best_time:.2f} seconds")
    
    # Compare with random baseline
    random_times = []
    for _ in range(100):
        random_seq = list(range(quantum_optimizer.num_tasks))
        np.random.shuffle(random_seq)
        random_times.append(quantum_optimizer.calculate_sequence_time(random_seq))
    
    avg_random_time = np.mean(random_times)
    improvement_vs_random = ((avg_random_time - best_time) / avg_random_time) * 100
    
    print(f"\nComparison with random baseline:")
    print(f"Random average: {avg_random_time:.2f} seconds")
    print(f"Quantum solution: {best_time:.2f} seconds")
    print(f"Improvement vs random: {improvement_vs_random:.1f}%")
    
    # Compare with Clarke-Wright baseline (simplified)
    def simple_clarke_wright(tasks):
        """Simple Clarke-Wright for comparison."""
        n = len(tasks)
        
        def distance(pos1, pos2):
            return max(abs(pos2[0] - pos1[0]), abs(pos2[1] - pos1[1]))
        
        depot = (1, 1)
        
        # Calculate savings
        savings = []
        for i in range(n):
            for j in range(i+1, n):
                pos_i = (tasks[i][1], tasks[i][2])
                pos_j = (tasks[j][1], tasks[j][2])
                depot_i = distance(depot, pos_i)
                depot_j = distance(depot, pos_j)
                task_ij = distance(pos_i, pos_j)
                savings.append((depot_i + depot_j - task_ij, i, j))
        
        # Sort by savings
        savings.sort(reverse=True)
        
        # Build sequence
        sequence = list(range(n))
        
        # Apply savings to improve sequence
        for save, i, j in savings[:min(5, len(savings))]:
            if i < len(sequence) and j < len(sequence):
                # Simple improvement: swap if it helps
                current_time = quantum_optimizer.calculate_sequence_time(sequence)
                new_sequence = sequence.copy()
                new_sequence[i], new_sequence[j] = new_sequence[j], new_sequence[i]
                new_time = quantum_optimizer.calculate_sequence_time(new_sequence)
                
                if new_time < current_time:
                    sequence = new_sequence
        
        return sequence
    
    # Run Clarke-Wright
    cw_sequence = simple_clarke_wright(all_tasks)
    cw_time = quantum_optimizer.calculate_sequence_time(cw_sequence)
    
    print(f"\nComparison with Clarke-Wright:")
    print(f"Clarke-Wright sequence: {[all_tasks[i][0] for i in cw_sequence]}")
    print(f"Clarke-Wright time: {cw_time:.2f} seconds")
    
    if best_time < cw_time:
        improvement_vs_cw = ((cw_time - best_time) / cw_time) * 100
        print(f"Quantum improvement vs Clarke-Wright: {improvement_vs_cw:.1f}%")
    else:
        degradation_vs_cw = ((best_time - cw_time) / cw_time) * 100
        print(f"Quantum degradation vs Clarke-Wright: {degradation_vs_cw:.1f}%")
except Exception as e:
    print(f'Error in cell: {e}')

[Timeout after 120s - cell wrapped for safety]

In [ ]:
try:
    # Visualize quantum annealing results
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot 1: Task locations and quantum optimal route
    colors = plt.cm.Set3(np.linspace(0, 1, len(best_sequence)))
    
    # Plot tasks
    for i, task in enumerate(all_tasks):
        marker = 's' if task[0].startswith('S') else 'o'
        color = 'red' if task[0].startswith('S') else 'blue'
        ax1.scatter(task[1], task[2], c=color, s=100, marker=marker, alpha=0.7)
        ax1.annotate(task[0], (task[1], task[2]), xytext=(5,5), textcoords='offset points')
    
    # Plot optimal route
    route_x = [quantum_optimizer.depot[0]]
    route_y = [quantum_optimizer.depot[1]]
    
    for task_idx in best_sequence:
        task = all_tasks[task_idx]
        route_x.append(task[1])
        route_y.append(task[2])
    
    route_x.append(quantum_optimizer.depot[0])
    route_y.append(quantum_optimizer.depot[1])
    
    ax1.plot(route_x, route_y, 'g--', alpha=0.5, linewidth=2, label='Quantum Optimal Route')
    ax1.scatter(quantum_optimizer.depot[0], quantum_optimizer.depot[1], c='green', s=200, marker='*', label='Depot', zorder=5)
    ax1.set_xlabel('X Position')
    ax1.set_ylabel('Y Position')
    ax1.set_title('Quantum Annealing Optimal Task Sequence')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Energy convergence
    ax2.plot(energy_history, 'b-', linewidth=2, label='Energy')
    ax2.set_xlabel('Sweep Number')
    ax2.set_ylabel('Energy')
    ax2.set_title('Quantum Annealing Energy Convergence')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Add best energy line
    ax2.axhline(y=best_energy, color='red', linestyle='--', alpha=0.7, label=f'Best Energy: {best_energy:.2f}')
    ax2.legend()
    
    # Plot 3: Performance comparison
    methods = ['Random', 'Clarke-Wright', 'Quantum']
    times = [avg_random_time, cw_time, best_time]
    colors_perf = ['red', 'orange', 'green']
    
    bars = ax3.bar(methods, times, color=colors_perf, alpha=0.7)
    ax3.set_ylabel('Total Time (seconds)')
    ax3.set_title('Performance Comparison')
    ax3.grid(True, alpha=0.3)
    
    # Add time labels on bars
    for bar, time in zip(bars, times):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                 f'{time:.1f}s', ha='center', va='bottom')
    
    # Plot 4: QUBO analysis
    # Create QUBO matrix visualization
    Q, linear = quantum_optimizer.create_qubo()
    Q_dense = Q + np.diag(linear)
    
    # Show only a portion for clarity
    show_size = min(10, Q_dense.shape[0])
    im = ax4.imshow(Q_dense[:show_size, :show_size], cmap='RdBu', aspect='auto')
    ax4.set_xlabel('QUBO Variable Index')
    ax4.set_ylabel('QUBO Variable Index')
    ax4.set_title(f'QUBO Matrix (First {show_size}×{show_size} variables)')
    plt.colorbar(im, ax=ax4)
    
    plt.tight_layout()
    plt.show()
    
    # Additional analysis
    print(f"\nQuantum Annealing Analysis:")
    print(f"QUBO variables: {len(linear)}")
    print(f"Non-zero QUBO elements: {np.count_nonzero(Q_dense)}")
    print(f"QUBO density: {np.count_nonzero(Q_dense) / (len(linear)**2):.2%}")
    print(f"Energy reduction: {energy_history[0] - best_energy:.2f}")
    print(f"Convergence sweeps: {len(energy_history)}")
    
    # Solution quality analysis
    print(f"\nSolution Quality:")
    print(f"Quantum solution: {best_time:.2f}s")
    print(f"Random baseline: {avg_random_time:.2f}s")
    print(f"Clarke-Wright: {cw_time:.2f}s")
    print(f"Quantum vs random improvement: {improvement_vs_random:.1f}%")
    
    if best_time < cw_time:
        print(f"Quantum vs Clarke-Wright improvement: {improvement_vs_cw:.1f}%")
    else:
        print(f"Quantum vs Clarke-Wright performance: {best_time/cw_time:.2f}x")
    
    # Constraint satisfaction
    decoded_seq, valid = quantum_optimizer.decode_solution(
        np.array([1 if i < len(best_sequence) and best_sequence[i] == j else 0 
                 for i in range(quantum_optimizer.num_tasks) for j in range(quantum_optimizer.num_tasks)])
    )
    
    print(f"\nConstraint Satisfaction:")
    print(f"Solution validity: {valid}")
    print(f"Sequence completeness: {len(decoded_seq) == quantum_optimizer.num_tasks}")
    print(f"No duplicate tasks: {len(set(decoded_seq)) == len(decoded_seq)}")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'best_sequence' is not defined...]

In [ ]:
try:
    # Scalability analysis
    print("\n" + "=" * 60)
    print("QUANTUM ANNEALING SCALABILITY ANALYSIS")
    print("=" * 60)
    
    # Test different problem sizes
    problem_sizes = [3, 4, 5, 6, 8, 10]
    optimization_times = []
    solution_qualities = []
    qubo_sizes = []
    
    for size in problem_sizes:
        print(f"\nTesting problem size: {size} tasks")
        
        # Create problem instance
        test_tasks = []
        for i in range(size):
            task_type = 'S' if i < size // 2 else 'R'
            x = random.randint(1, 10)
            y = random.randint(1, 8)
            priority = random.randint(1, 10)
            test_tasks.append((f'{task_type}{i+1}', x, y, priority))
        
        # Create optimizer
        test_optimizer = QuantumASRS(test_tasks, depot=(1,1))
        
        # Create QUBO
        Q, linear = test_optimizer.create_qubo()
        qubo_size = len(linear)
        qubo_sizes.append(qubo_size)
        
        # Run optimization (reduced sweeps for larger problems)
        original_sweeps = test_optimizer.annealing_schedule['num_sweeps']
        if size > 6:
            test_optimizer.annealing_schedule['num_sweeps'] = 500
        
        start_time = time.time()
        sequence, total_time, energy, history = test_optimizer.optimize()
        end_time = time.time()
        
        opt_time = end_time - start_time
        optimization_times.append(opt_time)
        
        # Calculate solution quality (vs random baseline)
        random_times = []
        for _ in range(20):  # Fewer random samples for speed
            random_seq = list(range(size))
            np.random.shuffle(random_seq)
            random_times.append(test_optimizer.calculate_sequence_time(random_seq))
        
        avg_random = np.mean(random_times)
        quality = ((avg_random - total_time) / avg_random) * 100
        solution_qualities.append(quality)
        
        print(f"  QUBO size: {qubo_size} variables")
        print(f"  Optimization time: {opt_time:.2f}s")
        print(f"  Solution time: {total_time:.2f}s")
        print(f"  Quality vs random: {quality:.1f}%")
        
        # Restore original sweeps
        test_optimizer.annealing_schedule['num_sweeps'] = original_sweeps
    
    # Visualize scalability results
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot 1: Optimization time vs problem size
    ax1.plot(problem_sizes, optimization_times, 'b-o', linewidth=2, markersize=8)
    ax1.set_xlabel('Problem Size (number of tasks)')
    ax1.set_ylabel('Optimization Time (seconds)')
    ax1.set_title('Quantum Annealing Scalability: Time')
    ax1.grid(True, alpha=0.3)
    ax1.set_yscale('log')
    
    # Plot 2: QUBO size vs problem size
    ax2.plot(problem_sizes, qubo_sizes, 'r-o', linewidth=2, markersize=8)
    ax2.set_xlabel('Problem Size (number of tasks)')
    ax2.set_ylabel('QUBO Variables')
    ax2.set_title('QUBO Formulation Complexity')
    ax2.grid(True, alpha=0.3)
    ax2.set_yscale('log')
    
    # Plot 3: Solution quality vs problem size
    ax3.plot(problem_sizes, solution_qualities, 'g-o', linewidth=2, markersize=8)
    ax3.set_xlabel('Problem Size (number of tasks)')
    ax3.set_ylabel('Improvement vs Random (%)')
    ax3.set_title('Solution Quality vs Problem Size')
    ax3.grid(True, alpha=0.3)
    ax3.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    
    # Plot 4: Complexity analysis
    theoretical_complexity = [n**2 for n in problem_sizes]  # QUBO complexity
    ax4.plot(problem_sizes, theoretical_complexity, 'purple', linewidth=2, label='Theoretical (n²)')
    ax4.plot(problem_sizes, qubo_sizes, 'orange', linewidth=2, label='Actual QUBO size')
    ax4.set_xlabel('Problem Size (number of tasks)')
    ax4.set_ylabel('Complexity')
    ax4.set_title('Complexity Analysis')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    ax4.set_yscale('log')
    
    plt.tight_layout()
    plt.show()
    
    # Scalability summary
    print(f"\nScalability Analysis Summary:")
    print(f"Problem sizes tested: {problem_sizes}")
    print(f"Average optimization time growth: {(optimization_times[-1] / optimization_times[0]):.1f}x")
    print(f"QUBO size growth: {(qubo_sizes[-1] / qubo_sizes[0]):.1f}x")
    print(f"Average solution quality: {np.mean(solution_qualities):.1f}%")
    print(f"Quality consistency: {np.std(solution_qualities):.1f}% std deviation")
    
    # Quantum advantage assessment
    print(f"\nQuantum Advantage Assessment:")
    if np.mean(solution_qualities) > 10:
        print(f"✅ Significant quantum advantage demonstrated")
        print(f"   Average improvement: {np.mean(solution_qualities):.1f}% over random")
    elif np.mean(solution_qualities) > 5:
        print(f"⚠️  Moderate quantum advantage")
        print(f"   Average improvement: {np.mean(solution_qualities):.1f}% over random")
    else:
        print(f"❌ Limited quantum advantage for current problem sizes")
        print(f"   Average improvement: {np.mean(solution_qualities):.1f}% over random")
    
    print(f"\nRecommendations:")
    print(f"• Quantum annealing shows promise for medium-scale problems (6-8 tasks)")
    print(f"• Larger problems may need hybrid quantum-classical approaches")
    print(f"• QUBO formulation complexity grows quadratically with problem size")
    print(f"• Real quantum hardware could provide speedup for larger instances")
except Exception as e:
    print(f'Error in cell: {e}')

[Timeout after 120s - cell wrapped for safety]

### Why This Tier Exists vs Earlier Tiers

**Tier 9: Quantum Leap** provides quantum computing capabilities with:
- **Quantum parallelism** - Simultaneous exploration of multiple solution states
- **Quantum tunneling** - Ability to escape local optima through quantum effects
- **QUBO formulation** - Natural mapping of optimization problems to quantum hardware
- **Exponential speedup potential** - Theoretical advantages for certain problem classes

### Pros / Cons vs Alternative Approaches

**Advantages:**
- ✅ **Quantum parallelism** - Explores multiple solutions simultaneously
- ✅ **Global optimization** - Less likely to get stuck in local optima
- ✅ **Natural QUBO mapping** - Many optimization problems map well to quantum annealing
- ✅ **Theoretical speedup** - Exponential advantages for specific problem types
- ✅ **Hardware acceleration** - Potential for massive parallel computation

**Disadvantages:**
- ❌ **Hardware accessibility** - Limited access to real quantum computers
- ❌ **Problem formulation complexity** - Requires QUBO transformation expertise
- ❌ **Noise and decoherence** - Current quantum hardware has reliability issues
- ❌ **Limited qubit count** - Problem sizes constrained by available qubits
- ❌ **Classical simulation overhead** - Simulated quantum annealing can be slow

### When to Use This Tier

**Use Quantum Leap when:**
- Problem can be naturally formulated as QUBO optimization
- Access to quantum annealing hardware is available
- Problem size fits within quantum hardware constraints
- Global optimization is critical and classical methods struggle
- Quantum advantage potential justifies the complexity
- Research and exploration of quantum computing applications

**Consider other tiers when:**
- Classical methods provide sufficient performance (use Tiers 1-6)
- Problem formulation doesn't map well to QUBO
- Quantum hardware access is limited
- Real-time performance requirements (use Tier 2 or 6)
- Simple optimization problems with known solutions (use Tier 1)